# Solution 06 — the joker and the critic

Two named agents. The joker's instructions require its tool, so its custom span shows up in every joker
trace. The critic returns a `Rating`, so the score is a checked `int`. The stretch (a parent span per
round) is included. With Logfire connected, open the **Agents view** to compare `joker` and `critic`.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
import logfire
from pydantic import BaseModel, Field
from pydantic_ai import Agent


class Rating(BaseModel):
    score: int = Field(ge=0, le=10)
    reason: str


joker = Agent(
    MODEL,
    name="joker",
    instructions=(
        "Always call get_setup_lines first. Then tell ONE short, clean programming joke "
        "that riffs on one of those lines. Reply with the joke only."
    ),
)


@joker.tool_plain
def get_setup_lines() -> list[str]:
    """Fetch classic setup lines for the joke to riff on."""
    with logfire.span("fetching setup lines"):
        return ["A bug walks into a bar...", "My code worked yesterday..."]


critic = Agent(
    MODEL,
    name="critic",
    output_type=Rating,
    instructions="Rate the joke you are given from 0 to 10 and give a one-sentence reason.",
)

In [ ]:
rounds = []
for n, topic in enumerate(["databases", "CSS", "Git"], start=1):
    with logfire.span("joke round {n}", n=n):
        joke = await joker.run(f"Tell me a joke about {topic}.")
        rating = await critic.run(joke.output)
    rounds.append((joke, rating))
    print(f"{n}. {joke.output}\n   → {rating.output.score}/10: {rating.output.reason}")

In [ ]:
for joke, rating in rounds:
    calls = [
        p.tool_name for m in joke.all_messages() for p in m.parts if p.part_kind == "tool-call"
    ]
    assert "get_setup_lines" in calls, calls  # the joker used its tool, so its span exists
    assert 0 <= rating.output.score <= 10, rating.output
assert len(rounds) == 3
print("✅ all checks passed")